# Notebook 3 — Model Training & Comparison

**What this notebook does:** Trains 4 classification models on the SMOTE-balanced,
scaled training data, evaluates each on the held-out test set, compares them, and
saves the best-performing model.

**Input:** `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

**Output:**
- `models/best_model.pkl`
- `outputs/model_comparison.csv`
- Charts saved to `outputs/charts/models/`

**Estimated run time:** ~1-2 minutes

In [1]:
# Imports and shared project utilities
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve
)

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import set_style, save_chart, save_dataframe, save_joblib_object, load_csv_safely, annotate_vertical_bars, project_path

set_style()

In [2]:
# Load the processed train/test splits produced by notebook 2
required_notebook = "02_preprocessing.ipynb"
X_train = load_csv_safely(project_path("data", "processed", "X_train.csv"), required_notebook)
X_test = load_csv_safely(project_path("data", "processed", "X_test.csv"), required_notebook)
y_train = load_csv_safely(project_path("data", "processed", "y_train.csv"), required_notebook)["Churn"]
y_test = load_csv_safely(project_path("data", "processed", "y_test.csv"), required_notebook)["Churn"]

print("X_train:", X_train.shape, " X_test:", X_test.shape)

X_train: (8260, 28)  X_test: (1407, 28)


## Helper function used for every model

Defined once and reused for all 4 models so the evaluation logic (metrics, confusion matrix, ROC, PR curve) isn't duplicated 4 times.

In [3]:
def evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    """Train a model, print its classification report, save its confusion
    matrix / ROC / PR curve charts, and return a dict of summary metrics."""
    file_slug = model_name.lower().replace(" ", "_")

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{'=' * 60}")
    print(f"MODEL: {model_name}")
    print(f"{'=' * 60}")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(12, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"], ax=ax)
    ax.set_title(f"Confusion Matrix — {model_name}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    save_chart(fig, project_path("outputs", "charts", "models", f"confusion_matrix_{file_slug}.png"))

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}", color="#4C72B0", linewidth=2)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
    ax.set_title(f"ROC Curve — {model_name}", fontsize=14, fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()
    save_chart(fig, project_path("outputs", "charts", "models", f"roc_curve_{file_slug}.png"))

    # Precision-Recall curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(recall_vals, precision_vals, color="#DD8452", linewidth=2)
    ax.set_title(f"Precision-Recall Curve — {model_name}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    save_chart(fig, project_path("outputs", "charts", "models", f"pr_curve_{file_slug}.png"))

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "AUC": auc_score,
        "Training_Time_seconds": training_time,
    }
    return model, metrics, fpr, tpr

## Train and evaluate all 4 models

In [4]:
# Model 1: Logistic Regression (baseline)
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg, log_reg_metrics, log_reg_fpr, log_reg_tpr = evaluate_model(
    log_reg, "Logistic Regression", X_train, y_train, X_test, y_test
)


MODEL: Logistic Regression
              precision    recall  f1-score   support

    No Churn       0.87      0.83      0.85      1033
       Churn       0.58      0.65      0.61       374

    accuracy                           0.78      1407
   macro avg       0.73      0.74      0.73      1407
weighted avg       0.79      0.78      0.79      1407

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/confusion_matrix_logistic_regression.png (50.2KB)


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/roc_curve_logistic_regression.png (84.4KB)


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/pr_curve_logistic_regression.png (69.6KB)


In [5]:
# Model 2: Random Forest
random_forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
random_forest, rf_metrics, rf_fpr, rf_tpr = evaluate_model(
    random_forest, "Random Forest", X_train, y_train, X_test, y_test
)


MODEL: Random Forest
              precision    recall  f1-score   support

    No Churn       0.86      0.83      0.84      1033
       Churn       0.57      0.61      0.59       374

    accuracy                           0.77      1407
   macro avg       0.71      0.72      0.72      1407
weighted avg       0.78      0.77      0.78      1407

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/confusion_matrix_random_forest.png (47.7KB)
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/roc_curve_random_forest.png (85.9KB)
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/pr_curve_random_forest.png (71.1KB)


In [6]:
# Model 3: XGBoost
xgboost_model = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42, eval_metric="logloss"
)
xgboost_model, xgb_metrics, xgb_fpr, xgb_tpr = evaluate_model(
    xgboost_model, "XGBoost", X_train, y_train, X_test, y_test
)


MODEL: XGBoost
              precision    recall  f1-score   support

    No Churn       0.85      0.82      0.83      1033
       Churn       0.54      0.59      0.56       374

    accuracy                           0.76      1407
   macro avg       0.69      0.70      0.70      1407
weighted avg       0.77      0.76      0.76      1407

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/confusion_matrix_xgboost.png (48.5KB)
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/roc_curve_xgboost.png (81.1KB)


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/pr_curve_xgboost.png (69.2KB)


In [7]:
# Model 4: Gradient Boosting (scikit-learn)
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42
)
gradient_boosting, gb_metrics, gb_fpr, gb_tpr = evaluate_model(
    gradient_boosting, "Gradient Boosting", X_train, y_train, X_test, y_test
)


MODEL: Gradient Boosting
              precision    recall  f1-score   support

    No Churn       0.87      0.81      0.84      1033
       Churn       0.56      0.66      0.60       374

    accuracy                           0.77      1407
   macro avg       0.71      0.73      0.72      1407
weighted avg       0.79      0.77      0.78      1407

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/confusion_matrix_gradient_boosting.png (50.2KB)
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/roc_curve_gradient_boosting.png (83.6KB)
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/pr_curve_gradient_boosting.png (71.1KB)


## Comparison charts

In [8]:
# All 4 ROC curves overlaid on one chart, labelled with model name and AUC
all_results = [
    ("Logistic Regression", log_reg_fpr, log_reg_tpr, log_reg_metrics["AUC"]),
    ("Random Forest", rf_fpr, rf_tpr, rf_metrics["AUC"]),
    ("XGBoost", xgb_fpr, xgb_tpr, xgb_metrics["AUC"]),
    ("Gradient Boosting", gb_fpr, gb_tpr, gb_metrics["AUC"]),
]

fig, ax = plt.subplots(figsize=(12, 6))
for name, fpr, tpr, auc_score in all_results:
    ax.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_score:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
ax.set_title("ROC Curves — All Models", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()

save_chart(fig, project_path("outputs", "charts", "models", "all_roc_curves.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/all_roc_curves.png (128.7KB)


In [9]:
# Bar chart comparing F1 scores across all 4 models, annotated with exact values
comparison_df = pd.DataFrame([log_reg_metrics, rf_metrics, xgb_metrics, gb_metrics])

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x="Model", y="F1", data=comparison_df, hue="Model", palette="Blues_d", legend=False, ax=ax)
ax.set_title("F1 Score Comparison Across Models", fontsize=14, fontweight="bold")
ax.set_ylabel("F1 Score")
annotate_vertical_bars(ax, fmt="{:.3f}")

save_chart(fig, project_path("outputs", "charts", "models", "f1_comparison.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/f1_comparison.png (54.1KB)


In [10]:
# Bar chart comparing AUC scores across all 4 models
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x="Model", y="AUC", data=comparison_df, hue="Model", palette="Oranges_d", legend=False, ax=ax)
ax.set_title("AUC Score Comparison Across Models", fontsize=14, fontweight="bold")
ax.set_ylabel("AUC Score")
annotate_vertical_bars(ax, fmt="{:.3f}")

save_chart(fig, project_path("outputs", "charts", "models", "auc_comparison.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/models/auc_comparison.png (58.9KB)


## Comparison table

In [11]:
# Build and save the full model comparison table
comparison_df = comparison_df[["Model", "Accuracy", "Precision", "Recall", "F1", "AUC", "Training_Time_seconds"]]
comparison_df = comparison_df.sort_values("AUC", ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))

save_dataframe(comparison_df, project_path("outputs", "model_comparison.csv"))

              Model  Accuracy  Precision   Recall       F1      AUC  Training_Time_seconds
Logistic Regression  0.783227   0.582734 0.649733 0.614412 0.831710               0.013433
  Gradient Boosting  0.771144   0.559091 0.657754 0.604423 0.826229               1.866020
            XGBoost  0.756930   0.538835 0.593583 0.564885 0.819031               0.100231
      Random Forest  0.773987   0.569307 0.614973 0.591260 0.813408               0.193884
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/model_comparison.csv (580.0B) — 4 rows, 7 columns


## Select and save the best model

In [12]:
# Select the best model by AUC — AUC is robust to the class imbalance in the
# test set (26.6% churn), unlike accuracy, which a model could game by
# predicting "no churn" for almost everyone and still scoring ~74%.
models_lookup = {
    "Logistic Regression": log_reg,
    "Random Forest": random_forest,
    "XGBoost": xgboost_model,
    "Gradient Boosting": gradient_boosting,
}

best_model_name = comparison_df.iloc[0]["Model"]
best_model_auc = comparison_df.iloc[0]["AUC"]
best_model = models_lookup[best_model_name]

save_joblib_object(best_model, project_path("models", "best_model.pkl"))

print(f"\nBest model: {best_model_name} with AUC of {best_model_auc:.3f}")
print(
    "\nAUC (Area Under the ROC Curve) was chosen as the model selection metric because "
    "the dataset is imbalanced (only ~26.6% of customers churn), which makes accuracy a "
    "misleading metric — a model that always predicts 'no churn' would still score close "
    "to 74% accuracy while being useless for the business. AUC instead measures how well "
    "the model ranks churners above non-churners across all possible decision thresholds, "
    "making it far more sensitive to a model's genuine discriminative power on the minority "
    "(churn) class, and it stays comparable even if the retention team later changes the "
    "probability threshold used to flag customers as high risk."
)

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/models/best_model.pkl (2.0KB)

Best model: Logistic Regression with AUC of 0.832

AUC (Area Under the ROC Curve) was chosen as the model selection metric because the dataset is imbalanced (only ~26.6% of customers churn), which makes accuracy a misleading metric — a model that always predicts 'no churn' would still score close to 74% accuracy while being useless for the business. AUC instead measures how well the model ranks churners above non-churners across all possible decision thresholds, making it far more sensitive to a model's genuine discriminative power on the minority (churn) class, and it stays comparable even if the retention team later changes the probability threshold used to flag customers as high risk.
